## ${\fbox{\textit{Stochastic Filtering Algorithms}}}$

$\underline{\textit{Arthur Cruz Shinzato}}$

$\fbox{\textit{List of Algorithms}}$

$a) \textit{Grid Filter}$

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Alphabet of symbols is given
# A = {1,2,3,4,5,6}

# Since the alphabet has a cardinality of 6, the number of states is naturally 6
N_states = 6

# ===================================================
# Input 1: Alphabet of possible symbols
# ===================================================

# Creating the alphabet
alphabet = np.arange(1, N_states + 1)

# Display the list of symbols
print("Symbols:")
print(alphabet)

Symbols:
[1 2 3 4 5 6]


In [2]:
# ================================================================
# Input 2: Initial probability of finding a symbol in the alphabet
# ================================================================

# Here we assumed a prior uniform distribution
p_0 = (1/N_states) * np.ones(N_states)
print("Vector with the initial probabilities for each symbol:")
print(p_0)

Vector with the initial probabilities for each symbol:
[0.16666667 0.16666667 0.16666667 0.16666667 0.16666667 0.16666667]


In [3]:
# ================================================================
# Input 3: Transition Probability Matrix
# ================================================================

# Note that the matrix is 6 x 6 since we have 6 possible outcomes in the alphabet
# The matrix is called stochastic because each column sums to 1
A = np.array([
    [0.4, 0.4, 0,   0,   0,   0],
    [0.6, 0.2, 0.4, 0,   0,   0],
    [0,   0.4, 0.2, 0.4, 0,   0],
    [0,   0,   0.4, 0.2, 0.4, 0],
    [0,   0,   0,   0.4, 0.2, 0.6],
    [0,   0,   0,   0,   0.4, 0.4]
])

print("Transition Probability Matrix:")
print(A)

Transition Probability Matrix:
[[0.4 0.4 0.  0.  0.  0. ]
 [0.6 0.2 0.4 0.  0.  0. ]
 [0.  0.4 0.2 0.4 0.  0. ]
 [0.  0.  0.4 0.2 0.4 0. ]
 [0.  0.  0.  0.4 0.2 0.6]
 [0.  0.  0.  0.  0.4 0.4]]


In [8]:
# ================================================================
# MODEL OF THE CHANNEL: AWGN (Aditive White Gaussian Noise)
# ================================================================

# X --> Symbols generated in the emitter
# Y --> Symbols received in the receiver
# W --> Noise affecting the channel
# Mathematically, Y = X + W, with W ~ N(0, sigma^2)

# ================================================================
# Input 4: Variance of the Noise
# ================================================================
sigma_W = 1

# The first step is to draw a sample from a multinomial distribution
index_0 = np.argmax(np.random.multinomial(1, p_0))
print("First state of the sequence:", index_0 + 1)

First state of the sequence: 5


In [9]:
# ================================================================
# Input 5: Number of Steps
# ================================================================

# Length of the sequence to be emulated
N_steps = 10000

# Structure to store the generated states
Vector_States_X = np.zeros(N_steps, dtype=int)

# Initial state already determined by the previous multinomial sampling
# Note that we need to add 1 since the index list belongs to the set (0,5) 
# and our alphabet belongs to the set (1,6)
Vector_States_X[0] = index_0 + 1

# ================================================================
# Emulation of the Generation of the Sequence of Symbols
# ================================================================

# Here we are emulating the Markov Chain using the matrix A
index = index_0
for i in range(1, N_steps):
    # Based on the index, we select a column of the matrix
    new_index = np.argmax(np.random.multinomial(1, A[:, index]))

    # Store the new state
    Vector_States_X[i] = new_index + 1

    # For the next iteration, we need to use the new index
    index = new_index

print("Sequence of States Generated")
print(Vector_States_X)

Sequence of States Generated
[5 4 3 ... 1 2 1]


In [10]:
# ================================================================
# Emulation of the contamination of the noise in the vector X
# ================================================================

# Emulation of the message contamination by AWGN
Vector_Y_received = Vector_States_X + sigma_W * np.random.randn(N_steps)

print("Sequence of states contaminated by noise")
print(Vector_Y_received)

Sequence of states contaminated by noise
[5.27243512 6.25398405 3.62980302 ... 0.13899411 3.60309191 1.51423954]


In [11]:
# ================================================================
# Determination of the Signal-to-Noise Ratio
# ================================================================

SNR = 10 * np.log10(np.sum(Vector_Y_received**2) /np.sum((Vector_Y_received - Vector_States_X)**2))
print(f"SNR with the selected noise: {SNR:.2f} dB")

SNR with the selected noise: 11.84 dB


In [12]:
# ================================
# Algorithm: Grid Filter
# Hidden Markov Model (HMM)
# ================================

# Here we start the process of estimating the sequence of symbols that were emitted by the
# transmitter based on the vector of observable measurements (Vector_Y_received)

# Vector to store the recovered symbols
Vector_Recovered_Symbols = np.zeros(N_steps)

# We are going to ignore the first 100 states since we want to consider the Markov Chain only
# in the steady-state scenario
L_steady_state = 100

# HMM Filter
p_old_posteriori = p_0.copy()

# Loop for estimating the hidden states
for i in range(L_steady_state, N_steps):

    # =====================================
    # Prediction Step: Obtaining p_{n|n-1}
    # =====================================

    # P_{n|n-1} = A P_{n-1|n-1}
    p_priori = A @ p_old_posteriori

    # =====================================
    # Update Step: Obtaining p_{n|n}
    # =====================================

    # For the update step, we need to compare the entire alphabet with the received symbol
    aux_Vector_comparison = Vector_Y_received[i] * np.ones_like(alphabet)

    # Observation Model: AWGN hypothesis
    beta = (1 / np.sqrt(2*np.pi*sigma_W**2)) * np.exp((-0.5 / sigma_W**2) * (aux_Vector_comparison - alphabet)**2)

    # Pointwise multiplication
    p_posteriori = beta * p_priori

    # Normalization to get a proper probability mass function (PMF)
    p_posteriori = p_posteriori / np.sum(p_posteriori)

    # Maximum a posteriori (MAP) criterion
    Vector_Recovered_Symbols[i] = np.argmax(p_posteriori) + 1

    # Final step: p_n|n becomes p_n-1|n-1
    p_old_posteriori = p_posteriori

# SER - Symbol Error Rate
# The first 100 symbols are discarded
SER_with_HMM_Filter = 1 - np.mean(Vector_Recovered_Symbols[L_steady_state:] == Vector_States_X[L_steady_state:])

print("Symbol Error Rate:", SER_with_HMM_Filter)
symbol_accuracy_rate = (1 - SER_with_HMM_Filter) * 100
print(f"Accuracy percentage = {symbol_accuracy_rate:.2f}%")

Symbol Error Rate: 0.46838383838383835
Accuracy percentage = 53.16%


In [13]:
# =================================================================
# In this second part of the project, we intend to evaluate the performance
# of the algorithm by varying the standard deviation values. As a metric
# of comparison, we are going to plot the estimation using the HMM filter
# and the estimation using the maximum likelihood algorithm (which does not
# consider a priori information about the source).
# =================================================================

# Thus, we are going to create a function that encapsulates the filter logic:

def Grid_Filter(L_steady_state, N_steps, A, p_0, Vector_Y_received, alphabet):
    # Same structure as before
    Vector_Recovered_Symbols = np.zeros(N_steps)
    p_old_posteriori = p_0.copy()

    # Loop for estimating the hidden states
    for i in range(L_steady_state, N_steps):

        # =====================================
        # Prediction Step: Obtaining p_{n|n-1}
        # =====================================

        p_priori = A @ p_old_posteriori

        # =====================================
        # Update Step: Obtaining p_{n|n}
        # =====================================

        aux_Vector_comparison = Vector_Y_received[i] * np.ones_like(alphabet)
        beta = (1 / np.sqrt(2*np.pi*sigma_W**2)) * np.exp((-0.5 / sigma_W**2) * (aux_Vector_comparison - alphabet)**2)
        p_posteriori = beta * p_priori
        p_posteriori = p_posteriori / np.sum(p_posteriori)
        Vector_Recovered_Symbols[i] = np.argmax(p_posteriori) + 1
        p_old_posteriori = p_posteriori

    return Vector_Recovered_Symbols

def Maximum_Likelihood_Algorithm(N_steps, Vector_Y_received, alphabet):
    # Same structure as before
    Vector_Recovered_Symbols = np.zeros(N_steps)

    for i in range(N_steps):
        Received_Symbol = Vector_Y_received[i] * np.ones_like(alphabet)

        # In this algorithm, we compare the distance between the received symbol and each symbol in the alphabet.
        # We then select the symbol according to the minimum distance criterion.
        Vector_Recovered_Symbols[i] = 1 + np.argmin(np.abs(Received_Symbol - alphabet))
    return Vector_Recovered_Symbols

def Markov_Chain_Generator(N_steps,N_states,A):
    
    # In this function we generate a markov chain given the transition matrix
    Markov_Chain_Outcomes = np.zeros(N_steps, dtype=int)
    
    # Initial state determination by multinomial sampling
    index_0 = np.argmax(np.random.multinomial(1, 1/N_states * np.ones(N_states)))
    Markov_Chain_Outcomes[0] = index_0 + 1
    index = index_0

    # Emulation of the sequence generation of symbols
    for i in range(1, N_steps):
        new_index = np.argmax(np.random.multinomial(1, A[:, index]))
        Markov_Chain_Outcomes[i] = new_index + 1
        index = new_index

    return Markov_Chain_Outcomes

In [ ]:
# Configuration to render all text and formulas using pure LaTeX
plt.rc('text', usetex=True)
plt.rc('font', family='serif')

# =================================================================
# Simulation Parameters and Noise Setup
# =================================================================

# Number of noise points to evaluate
N_noise_points = 5000
sigma_0 = 10**(-3)
sigma_f = 2

# Array of standard deviations (noise levels)
Vector_Noise_points = np.linspace(sigma_0, sigma_f, N_noise_points)

# Arrays to store the Symbol Error Rate (SER) for each algorithm
vector_SER_HMM = np.zeros(N_noise_points)
vector_SER_ML = np.zeros(N_noise_points)

# Outer loop to evaluate performance under different noise conditions
for j in range(len(Vector_Noise_points)):
    # Update the noise standard deviation for the current simulation step
    sigma_W = Vector_Noise_points[j]
  
    # =================================================================
    # Markov Chain Generator
    # =================================================================
    Vector_States_X = Markov_Chain_Generator(N_steps,N_states,A)

    # =================================================================
    # Channel Emulation (AWGN Noise Contamination)
    # =================================================================
    Y_rec = Vector_States_X + sigma_W * np.random.randn(N_steps)

    # =================================================================
    # Signal Estimation Using the Defined Functions
    # =================================================================
    # 1. Grid/HMM Filter Estimation
    vet_recovered_HMM = Grid_Filter(L_steady_state, N_steps, A, p_0, Y_rec, alphabet)
    
    # 2. Maximum Likelihood Estimation
    vet_recovered_ML = Maximum_Likelihood_Algorithm(N_steps, Y_rec, alphabet)

    # =================================================================
    # SER Calculation (Ignoring the first samples)
    # =================================================================
    # Note that the first 100 symbols are discarded for the steady-state scenario
    vector_SER_HMM[j] = 1 - np.mean(vet_recovered_HMM[L_steady_state:] == Vector_States_X[L_steady_state:])
    
    # ML SER calculation
    vector_SER_ML[j] = 1 - np.mean(vet_recovered_ML[L_steady_state:] == Vector_States_X[L_steady_state:])
    
# =================================================================
# Data Plotting (Using LaTeX formatting)
# =================================================================
plt.figure(figsize=(7, 5))

# Plotting both curves for comparison
plt.semilogy(Vector_Noise_points, vector_SER_HMM, label=r'Grid Filter (HMM)', linewidth=2)
plt.semilogy(Vector_Noise_points, vector_SER_ML, label=r'Maximum Likelihood (ML)', linestyle='--', linewidth=2)

# Labels and Titles using LaTeX syntax
plt.xlabel(r'Noise Standard Deviation ($\sigma$)', fontsize=12)
plt.ylabel(r'Symbol Error Rate ($\mathrm{SER}$)', fontsize=12)
plt.title(r'Performance Comparison: HMM Filter vs. Maximum Likelihood', fontsize=13)

# Grid and Legend adjustments
plt.grid(True, which="both", linestyle="--", alpha=0.7)
plt.legend(loc="best", fontsize=11)
plt.tight_layout()
plt.show()